# SemIf-OpenJev Decision Model on Snowflake

Deploy [SemIf-OpenJev](https://github.com/TheoLeeCJ/SemIf-OpenJev) — a semantic inference scoring model using **Qwen/Qwen3.5-4B (frozen)** — as a real-time SPCS inference service via Snowflake Model Registry.

**How it works:** Given a text `state`, a `question`, and 2–16 `options`, SemIf returns a probability for each option in a single forward pass. No text generation, no classification head — just probability scoring.

### Prerequisites
- **Roles:** SYSADMIN (most operations) + ACCOUNTADMIN (external access integration only)
- **GPU compute pool:** GPU_NV_S (single A10G, 22 GB VRAM) — model runs at bf16
- **Notebook service:** Must have external access to PyPI + HuggingFace + GitHub (configured below)
- **Time:** ~15–25 min end-to-end (model download ~5 min, log_model upload ~5 min, SPCS image build ~5–10 min)
- **No GitHub API key needed** — just raw HTTPS file downloads from a public repo
- **No HuggingFace API key needed** — Qwen3.5-4B is a public model

In [ ]:
-- Step 1: Infrastructure — database, schema, warehouse, compute pool, image repo, network rule
USE ROLE SYSADMIN;

CREATE DATABASE IF NOT EXISTS JEV_DEMO;
CREATE SCHEMA IF NOT EXISTS JEV_DEMO.INFERENCE;

CREATE WAREHOUSE IF NOT EXISTS OPENJEV_WH
    WAREHOUSE_SIZE = 'XSMALL'
    AUTO_SUSPEND = 300
    AUTO_RESUME = TRUE;

USE DATABASE JEV_DEMO;
USE SCHEMA INFERENCE;
USE WAREHOUSE OPENJEV_WH;

-- GPU compute pool for the inference service
CREATE COMPUTE POOL IF NOT EXISTS OPENJEV_GPU_POOL
    MIN_NODES = 1
    MAX_NODES = 2
    INSTANCE_FAMILY = GPU_NV_S
    AUTO_RESUME = TRUE
    AUTO_SUSPEND_SECS = 600;

-- Image repository for the auto-built container image
CREATE IMAGE REPOSITORY IF NOT EXISTS JEV_DEMO.INFERENCE.IMAGES;

-- Network rule: allow egress to PyPI, GitHub (SemIf source), and HuggingFace (model weights)
-- HuggingFace uses many CDN/CAS subdomains across multiple levels. We use wildcards to
-- future-proof against new endpoints, plus explicit entries where wildcards can't reach.
CREATE OR REPLACE NETWORK RULE JEV_DEMO.INFERENCE.OPENJEV_NETWORK_RULE
    MODE = EGRESS
    TYPE = HOST_PORT
    VALUE_LIST = (
        'pypi.org',
        'files.pythonhosted.org',
        'raw.githubusercontent.com',
        'huggingface.co',
        '*.huggingface.co',
        '*.huggingface.com',
        '*.hf.co',
        '*.xethub.hf.co',
        '*.cdn.hf.co',
        '*.aws.cdn.hf.co',
        'transfer.xethub.hf.co',
        'cas-server.xethub.hf.co',
        'cas-bridge.xethub.hf.co'
    );

In [ ]:
%%sql -r eai_result
-- Step 2: External Access Integration (requires ACCOUNTADMIN or CREATE INTEGRATION privilege)
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION OPENJEV_NETWORK_ACCESS
    ALLOWED_NETWORK_RULES = (JEV_DEMO.INFERENCE.OPENJEV_NETWORK_RULE)
    ENABLED = TRUE;

GRANT USAGE ON INTEGRATION OPENJEV_NETWORK_ACCESS TO ROLE SYSADMIN;

-- Switch back to SYSADMIN for the rest of the notebook
USE ROLE SYSADMIN;
USE DATABASE JEV_DEMO;
USE SCHEMA INFERENCE;

## Notebook Service Configuration

**Before running the cells below, you must create or edit your notebook service with the correct settings.**

This notebook requires:
- **Compute pool:** `OPENJEV_GPU_POOL` (created in Step 1 above) — a GPU is required for model inference
- **External Access Integrations:** `OPENJEV_EXTERNAL_ACCESS` (created in Step 2 above) — needed to download packages, source code, and model weights

**Steps:**
1. Click the **Connected** dropdown at the top of this notebook
2. Select **Create service** (or edit an existing GPU service)
3. Set the **Compute pool** to `OPENJEV_GPU_POOL`
4. Under **External Access**, add both:
   - `PYPI_ACCESS_INTEGRATION` (for pip installs)
   - `OPENJEV_EXTERNAL_ACCESS` (for GitHub and HuggingFace access)
5. Click **Create** and wait for the service to reach `RUNNING` status (~1-5 minutes for GPU)

**Example configuration:**

![Notebook Service Configuration](example_config.png)

> **Run the validation cell below to confirm your service is correctly configured before proceeding.**

In [ ]:
%pip install huggingface_hub snowflake-ml-python --quiet

In [ ]:
import urllib.request
import os
import sys

# Download SemIf-OpenJev source files (MIT license) from GitHub
# No API key needed — plain HTTPS download of raw files from a public repo
COMMIT = "1f2dea3e25379f9dfc98cb83c324f00ab5deda37"
BASE_URL = f"https://raw.githubusercontent.com/TheoLeeCJ/SemIf-OpenJev/{COMMIT}/src/semif_phase1"

VENDOR_DIR = "/tmp/semif_vendored"
os.makedirs(VENDOR_DIR, exist_ok=True)

# Empty __init__.py to make it an importable package
with open(os.path.join(VENDOR_DIR, "__init__.py"), "w") as f:
    pass

for filename in ["core.py", "direct.py"]:
    url = f"{BASE_URL}/{filename}"
    dest = os.path.join(VENDOR_DIR, filename)
    print(f"Downloading {filename} from GitHub...")
    urllib.request.urlretrieve(url, dest)
    print(f"  -> {dest} ({os.path.getsize(dest):,} bytes)")

# Ensure /tmp is on the Python path so semif_vendored is importable
if "/tmp" not in sys.path:
    sys.path.insert(0, "/tmp")

print("\nSemIf source files ready.")

In [ ]:
from huggingface_hub import snapshot_download

# Download Qwen3.5-4B weights (~9.3 GB) — public model, no API key needed
# Pinned to the exact revision used by SemIf-OpenJev
print("Downloading Qwen/Qwen3.5-4B weights (~9.3 GB)...")
print("This typically takes 3-5 minutes depending on network speed.\n")

snapshot_download(
    "Qwen/Qwen3.5-4B",
    revision="851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a",
    local_dir="/tmp/qwen35_4b",
)

print("\nModel weights downloaded to /tmp/qwen35_4b")

In [ ]:
import json
import pandas as pd
from snowflake.ml.model import custom_model

BASE_REVISION = "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"

class OpenJevDecisionModel(custom_model.CustomModel):
    """SemIf scoring: frozen Qwen3.5-4B + SemIf probability scoring."""

    def __init__(self, context: custom_model.ModelContext = None) -> None:
        super().__init__(context)
        self._model = None
        self._tokenizer = None
        self._metadata = None

    def _ensure_loaded(self) -> None:
        if self._model is not None:
            return
        from semif_vendored.core import load_causal_model
        local_weights = self.context.artifacts["base_model"]
        self._model, self._tokenizer, self._metadata = load_causal_model(
            local_weights, BASE_REVISION, device="auto", dtype="bfloat16"
        )

    @custom_model.inference_api
    def predict(self, input_df: pd.DataFrame) -> pd.DataFrame:
        self._ensure_loaded()
        from semif_vendored.direct import score
        out_ids, choices, options_out = [], [], []
        for _, r in input_df.iterrows():
            options = json.loads(r["options_json"])
            row = {
                "id": str(r["id"]),
                "state": r["state"],
                "question": r["question"],
                "options": options,
            }
            result = score(self._model, self._tokenizer, row, self._metadata)
            best_idx = max(
                range(len(result["probabilities"])),
                key=result["probabilities"].__getitem__,
            )
            out_ids.append(result["id"])
            choices.append(result["option_ids"][best_idx])
            options_out.append(
                json.dumps(
                    [{"id": oid, "probability": p}
                     for oid, p in zip(result["option_ids"], result["probabilities"])]
                )
            )
        return pd.DataFrame({"id": out_ids, "choice": choices, "options_json": options_out})

print("OpenJevDecisionModel class defined.")
print("Input:  id, state (text), question, options_json (JSON array of {id, description})")
print("Output: id, choice (best option id), options_json (JSON array of {id, probability})")

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry
from snowflake.ml.model import model_signature as ms

session = get_active_session()
session.sql("USE DATABASE JEV_DEMO").collect()
session.sql("USE SCHEMA INFERENCE").collect()

signature = ms.ModelSignature(
    inputs=[
        ms.FeatureSpec(name="id", dtype=ms.DataType.STRING),
        ms.FeatureSpec(name="state", dtype=ms.DataType.STRING),
        ms.FeatureSpec(name="question", dtype=ms.DataType.STRING),
        ms.FeatureSpec(name="options_json", dtype=ms.DataType.STRING),
    ],
    outputs=[
        ms.FeatureSpec(name="id", dtype=ms.DataType.STRING),
        ms.FeatureSpec(name="choice", dtype=ms.DataType.STRING),
        ms.FeatureSpec(name="options_json", dtype=ms.DataType.STRING),
    ],
)

model = OpenJevDecisionModel(
    context=custom_model.ModelContext(artifacts={"base_model": "/tmp/qwen35_4b"})
)

reg = Registry(session=session, database_name="JEV_DEMO", schema_name="INFERENCE")

print("Logging model to registry (uploading ~9 GB of bundled weights)...")
print("This typically takes 3-5 minutes.\n")

# Use conda_dependencies instead of pip_requirements to avoid the +cu124 local
# build tag that the registry auto-detects from the installed torch package.
# Conda versions don't carry CUDA suffixes, sidestepping the issue entirely.
mv = reg.log_model(
    model=model,
    model_name="OPENJEV",
    version_name="V1",
    signatures={"predict": signature},
    code_paths=["/tmp/semif_vendored"],
    conda_dependencies=[
        "conda-forge::pytorch",
        "conda-forge::transformers==5.17.0",
        "conda-forge::accelerate",
        "conda-forge::flash-linear-attention==0.5.2",
    ],
    target_platforms=["SNOWPARK_CONTAINER_SERVICES"],
    options={"cuda_version": "12.4"},
)
print(f"Model logged: {mv.model_name} / {mv.version_name}")

print("\nCreating inference service (auto image build + deploy)...")
print("First-time build may anywhere from 30 minutes -> 1 hour.\n")

mv.create_service(
    service_name="OPENJEV_SVC",
    service_compute_pool="OPENJEV_GPU_POOL",
    image_repo="JEV_DEMO.INFERENCE.IMAGES",
    gpu_requests="1",
    ingress_enabled=True,
    max_instances=1,
)
print("Service creation initiated!")
print("Run the next SQL cell to check status — wait for RUNNING before testing.")

In [ ]:
%%sql -r service_status
-- Re-run this cell until STATUS = 'RUNNING'
SHOW SERVICES IN SCHEMA JEV_DEMO.INFERENCE;

## Testing

Once `OPENJEV_SVC` shows status **RUNNING** above, run the cells below to test the model.

The `!PREDICT` service function takes 4 arguments:
- `id` — row identifier
- `state` — the text to evaluate
- `question` — what to decide about the text
- `options_json` — JSON array of `{"id": ..., "description": ...}` (2–16 options)

In [ ]:
-- Single prediction: restaurant review sentiment
SELECT JEV_DEMO.INFERENCE.OPENJEV_SVC!PREDICT(
    'r1',
    'The waiter forgot our order twice and the pasta was cold when it finally arrived. Never coming back.',
    'Was this a good or bad restaurant experience?',
    '[{"id":"good","description":"The guest had a good dining experience and would recommend the restaurant."},{"id":"bad","description":"The guest had a bad dining experience and would not recommend the restaurant."}]'
) AS result;

In [ ]:
-- Create sample restaurant reviews for batch testing
CREATE OR REPLACE TABLE JEV_DEMO.INFERENCE.RESTAURANT_REVIEWS (
    review_id STRING,
    review    STRING
);

INSERT INTO JEV_DEMO.INFERENCE.RESTAURANT_REVIEWS (review_id, review) VALUES
    ('r1', 'Incredible tasting menu — every course was a surprise. The sommelier paired wines perfectly. Already made a reservation for next month.'),
    ('r2', 'The waiter forgot our order twice and the pasta was cold when it finally arrived. Never coming back.'),
    ('r3', 'Great ambiance and creative cocktails, but the main course was oversalted and the portions were tiny for the price.'),
    ('r4', 'Oh sure, I loved waiting 90 minutes for a table with a reservation. Truly a highlight of the evening.'),
    ('r5', 'Nothing fancy, but solid comfort food, quick service, and very reasonable prices. Perfect weeknight spot.'),
    ('r6', 'They messed up our reservation and we almost left, but the chef sent out a complimentary appetizer spread that was honestly the best thing we ate all night.');

In [ ]:
-- Batch scoring: classify all reviews with probability breakdown
-- !PREDICT returns an OBJECT (VARIANT) — do NOT wrap in PARSE_JSON.
-- Only the nested "options_json" field (a JSON string) needs PARSE_JSON.
SELECT
    review_id,
    review,
    decision:"choice"::STRING AS verdict,
    PARSE_JSON(decision:"options_json"::STRING)[0]:"probability"::FLOAT AS p_good
FROM (
    SELECT
        review_id,
        review,
        JEV_DEMO.INFERENCE.OPENJEV_SVC!PREDICT(
            review_id,
            review,
            'Was this a good or bad restaurant experience?',
            '[{"id":"good","description":"The guest had a good dining experience and would recommend the restaurant."},{"id":"bad","description":"The guest had a bad dining experience and would not recommend the restaurant."}]'
        ) AS decision
    FROM JEV_DEMO.INFERENCE.RESTAURANT_REVIEWS
)
ORDER BY review_id;

### Expected Results

| review_id | verdict | p_good | Review (summary) |
|-----------|---------|---------|------------------|
| r1 | good | ~0.999 | Incredible tasting menu, perfect wine pairings |
| r2 | bad | ~0.001 | Forgotten order, cold pasta |
| r3 | bad | ~0.16 | Great ambiance but oversalted and tiny portions |
| r4 | bad | ~0.007 | Sarcastic about 90-minute wait with reservation |
| r5 | good | ~0.999 | Solid comfort food, quick service, good value |
| r6 | good | ~0.977 | Messed up reservation, but amazing complimentary appetizers |

Notably: r3 and r6 show the model weighing **mixed sentiment** sensibly — r3 ("great ambiance" + "oversalted and tiny portions") leans bad; r6 ("messed up reservation" + "complimentary appetizer spread was the best thing we ate") leans good since the recovery outweighs the rocky start.

## Cleanup

Uncomment and run the cell below to tear down **all** demo resources. Only do this when you are done testing.

In [ ]:
%%sql -r cleanup_result
-- DROP SERVICE IF EXISTS JEV_DEMO.INFERENCE.OPENJEV_SVC;
-- DROP MODEL IF EXISTS JEV_DEMO.INFERENCE.OPENJEV;
-- ALTER COMPUTE POOL OPENJEV_GPU_POOL STOP ALL;
-- DROP COMPUTE POOL IF EXISTS OPENJEV_GPU_POOL;
-- DROP WAREHOUSE IF EXISTS OPENJEV_WH;
-- DROP DATABASE IF EXISTS JEV_DEMO;  -- removes DB, schema, tables, stages, image repo, network rule